In [47]:
import os
import re
import hashlib
from pathlib import Path
from email import policy
from email.parser import BytesParser
from email.utils import getaddresses, parsedate_to_datetime
import pandas as pd

In [48]:
# =========================
# 1. 基础清洗函数
# =========================
def clean_text(text):
    """Basic body text cleaning."""
    if text is None:
        return ""
    text = str(text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

In [49]:
def safe_str(x):
    if x is None:
        return ""
    return str(x).strip()

In [50]:
def make_stable_id(file_path):
    """
    Generate a stable integer-like id from file path.
    Matches your PST table style better than raw path string.
    """
    h = hashlib.md5(str(file_path).encode("utf-8")).hexdigest()
    return int(h[:12], 16)  # long but still manageable

In [51]:
# =========================
# 2. Header / 地址提取
# =========================
def extract_addresses(header_value):
    """
    Parse addresses from To / Cc / From etc.
    Returns:
        names_str, emails_str, raw_pairs
    """
    if not header_value:
        return "", "", []

    pairs = getaddresses([header_value])

    names = []
    emails = []
    clean_pairs = []

    for name, email_addr in pairs:
        name = safe_str(name)
        email_addr = safe_str(email_addr)

        if name:
            names.append(name)
        if email_addr:
            emails.append(email_addr)

        if name or email_addr:
            clean_pairs.append((name, email_addr))

    names_str = "; ".join(names)
    emails_str = "; ".join(emails)

    return names_str, emails_str, clean_pairs

In [52]:
def extract_raw_headers(msg):
    """
    Convert all headers into a raw string similar to PST 'headers'.
    """
    try:
        return "".join(f"{k}: {v}\n" for k, v in msg.items())
    except Exception:
        return ""

In [53]:
# =========================
# 3. 正文提取
# =========================
def get_body_from_eml(msg):
    """
    Prefer text/plain.
    If no plain text exists, fallback to text/html.
    Skip attachments.
    """
    text_body = ""
    html_body = ""

    try:
        if msg.is_multipart():
            for part in msg.walk():
                content_type = part.get_content_type()
                content_disposition = str(part.get("Content-Disposition", "")).lower()

                # skip attachments
                if "attachment" in content_disposition:
                    continue

                try:
                    payload = part.get_payload(decode=True)
                except Exception:
                    payload = None

                if payload is None:
                    continue

                charset = part.get_content_charset() or "utf-8"

                try:
                    decoded = payload.decode(charset, errors="ignore")
                except Exception:
                    try:
                        decoded = payload.decode("utf-8", errors="ignore")
                    except Exception:
                        decoded = ""

                if content_type == "text/plain" and not text_body:
                    text_body = decoded
                elif content_type == "text/html" and not html_body:
                    html_body = decoded
        else:
            content_type = msg.get_content_type()

            try:
                payload = msg.get_payload(decode=True)
            except Exception:
                payload = None

            if payload is not None:
                charset = msg.get_content_charset() or "utf-8"
                try:
                    decoded = payload.decode(charset, errors="ignore")
                except Exception:
                    try:
                        decoded = payload.decode("utf-8", errors="ignore")
                    except Exception:
                        decoded = ""
            else:
                decoded = ""

            if content_type == "text/plain":
                text_body = decoded
            elif content_type == "text/html":
                html_body = decoded

    except Exception:
        pass

    body = text_body if text_body else html_body
    return clean_text(body)

In [54]:
# =========================
# 4. 附件提取
# =========================
def extract_attachments(msg):
    """
    Returns a list of attachment file names.
    """
    attachments = []

    try:
        for part in msg.walk():
            content_disposition = str(part.get("Content-Disposition", "")).lower()
            if "attachment" in content_disposition:
                filename = part.get_filename()
                if filename:
                    attachments.append(str(filename))
    except Exception:
        pass

    return attachments

In [55]:
# =========================
# 5. 日期解析
# =========================
def parse_email_date(date_value):
    """
    Convert email Date header into pandas datetime.
    """
    if not date_value:
        return pd.NaT

    try:
        dt = parsedate_to_datetime(str(date_value))
        if dt is None:
            return pd.NaT

        # remove timezone to align more easily with your PST datetime64[us]
        if getattr(dt, "tzinfo", None) is not None:
            dt = dt.astimezone().replace(tzinfo=None)

        return pd.to_datetime(dt)
    except Exception:
        return pd.NaT

In [56]:
# =========================
# 6. 单个 EML 解析
# =========================
def parse_single_eml(file_path):
    """
    Parse one .eml file into a dict matching your PST schema.
    """
    file_path = Path(file_path)

    with open(file_path, "rb") as f:
        msg = BytesParser(policy=policy.default).parse(f)

    # raw headers
    headers = extract_raw_headers(msg)

    # sender
    from_header = safe_str(msg.get("From", ""))
    sender_names, sender_emails, from_pairs = extract_addresses(from_header)

    sender = sender_names if sender_names else ""
    sender_email = sender_emails if sender_emails else ""

    # to / cc
    to_header = safe_str(msg.get("To", ""))
    cc_header = safe_str(msg.get("Cc", ""))

    to_names, to_emails, to_pairs = extract_addresses(to_header)
    cc_names, cc_emails, cc_pairs = extract_addresses(cc_header)

    # recipient aggregation
    all_recipient_names = []
    all_recipient_emails = []

    for name, email_addr in to_pairs + cc_pairs:
        if name:
            all_recipient_names.append(name)
        if email_addr:
            all_recipient_emails.append(email_addr)

    recipient_names = "; ".join(all_recipient_names)
    recipient_emails = "; ".join(all_recipient_emails)

    # body
    body = get_body_from_eml(msg)

    # attachments
    attachment_names = extract_attachments(msg)
    has_attachments = len(attachment_names) > 0
    attachment_count = len(attachment_names)

    # date
    date_parsed = parse_email_date(msg.get("Date", ""))

    # thread headers
    message_id = safe_str(msg.get("Message-ID", ""))
    in_reply_to = safe_str(msg.get("In-Reply-To", ""))
    references = safe_str(msg.get("References", ""))

    record = {
        "id": make_stable_id(file_path),
        "subject": safe_str(msg.get("Subject", "")),
        "sender": sender,
        "sender_email": sender_email,
        "to": to_header,
        "cc": cc_header,
        "date": date_parsed,
        "body": body,
        "recipient_emails": recipient_emails,
        "recipient_names": recipient_names,
        "source_pst": "",                       # keep aligned with PST schema
        "folder_path": str(file_path.parent),
        "has_attachments": has_attachments,
        "attachment_count": attachment_count,
        "attachment_names": attachment_names,
        "headers": headers,
        "message_id": message_id,
        "in_reply_to": in_reply_to,
        "references": references,
        "source_type": "eml",
        "source_file": file_path.name,
    }

    return record

In [57]:
# =========================
# 7. 批量解析文件夹
# =========================
def parse_eml_folder(folder_path, progress_every=500):
    """
    Recursively parse all .eml files under a folder.
    Returns:
        df_records, df_failed
    """
    folder_path = Path(folder_path)
    eml_files = [
    p for p in folder_path.rglob("*")
    if p.is_file() and p.suffix.lower() == ".eml"
    ]

    print(f"Found {len(eml_files)} .eml files in: {folder_path}")

    records = []
    failed_files = []

    for i, file_path in enumerate(eml_files, 1):
        try:
            record = parse_single_eml(file_path)
            records.append(record)
        except Exception as e:
            failed_files.append({
                "source_file": str(file_path),
                "error": str(e)
            })

        if i % progress_every == 0:
            print(f"Processed {i}/{len(eml_files)}")

    df_records = pd.DataFrame(records)
    df_failed = pd.DataFrame(failed_files)

    return df_records, df_failed

In [58]:
# =========================
# 8. 对齐到 PST schema（可选）
# =========================
def align_to_target_schema(df, target_columns):
    """
    Make sure df has all target columns in the same order.
    Missing columns are filled with empty string / NA.
    """
    df = df.copy()

    for col in target_columns:
        if col not in df.columns:
            if col == "date":
                df[col] = pd.NaT
            elif col == "has_attachments":
                df[col] = False
            elif col == "attachment_count":
                df[col] = 0
            elif col == "attachment_names":
                df[col] = [[] for _ in range(len(df))]
            else:
                df[col] = ""

    return df[target_columns]

In [59]:
# =========================
# 9. 主流程示例
# =========================
# 你把这两个路径改成你自己的
folder1 = "/Users/promab/anaconda_projects/email_agent/data/raw/eml/2015"
folder2 = "/Users/promab/anaconda_projects/email_agent/data/raw/eml/mail copy by Dewen"

df_eml_1, df_fail_1 = parse_eml_folder(folder1, progress_every=500)
df_eml_2, df_fail_2 = parse_eml_folder(folder2, progress_every=500)

# 合并两个文件夹结果
df_eml_all = pd.concat([df_eml_1, df_eml_2], ignore_index=True)
df_eml_fail_all = pd.concat([df_fail_1, df_fail_2], ignore_index=True)

print("EML total shape:", df_eml_all.shape)
print("Failed files shape:", df_eml_fail_all.shape)

Found 4729 .eml files in: /Users/promab/anaconda_projects/email_agent/data/raw/eml/2015
Processed 500/4729
Processed 1000/4729
Processed 1500/4729
Processed 2000/4729
Processed 2500/4729
Processed 3000/4729
Processed 3500/4729
Processed 4000/4729
Processed 4500/4729
Found 9212 .eml files in: /Users/promab/anaconda_projects/email_agent/data/raw/eml/mail copy by Dewen
Processed 500/9212
Processed 1000/9212
Processed 1500/9212
Processed 2000/9212
Processed 2500/9212
Processed 3000/9212
Processed 3500/9212
Processed 4000/9212
Processed 4500/9212
Processed 5000/9212
Processed 5500/9212
Processed 6000/9212
Processed 6500/9212
Processed 7000/9212
Processed 7500/9212
Processed 8000/9212
Processed 8500/9212
Processed 9000/9212
EML total shape: (13941, 21)
Failed files shape: (0, 0)


In [64]:
# =========================
# 10. 如果你要和 PST 对齐，推荐先定义最终列顺序
# =========================
target_columns = [
    "id",
    "subject",
    "sender",
    "sender_email",
    "to",
    "cc",
    "date",
    "body",
    "recipient_emails",
    "recipient_names",
    "source_pst",
    "folder_path",
    "has_attachments",
    "attachment_count",
    "attachment_names",
    "headers",
    "message_id",
    "in_reply_to",
    "references",
    "source_type",
    "source_file",
]

df_eml_all = align_to_target_schema(df_eml_all, target_columns)

# 看看结果
print(df_eml_all.info())
display(df_eml_all.head())

# =========================
# 11. 保存
# =========================
out_dir = Path("/Users/promab/anaconda_projects/email_agent/data/processed/eml_parsed")
out_dir.mkdir(parents=True, exist_ok=True)

df_eml_all.to_parquet(out_dir / "parsed_eml_all.parquet", index=False)

print("Saved parsed EML files.")

<class 'pandas.DataFrame'>
RangeIndex: 13941 entries, 0 to 13940
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   id                13941 non-null  int64         
 1   subject           13941 non-null  str           
 2   sender            13941 non-null  str           
 3   sender_email      13941 non-null  str           
 4   to                13941 non-null  str           
 5   cc                13941 non-null  str           
 6   date              13795 non-null  datetime64[us]
 7   body              13941 non-null  str           
 8   recipient_emails  13941 non-null  str           
 9   recipient_names   13941 non-null  str           
 10  source_pst        13941 non-null  str           
 11  folder_path       13941 non-null  str           
 12  has_attachments   13941 non-null  bool          
 13  attachment_count  13941 non-null  int64         
 14  attachment_names  13941 non-null 

,id,subject,sender,sender_email,to,cc,date,body,recipient_emails,recipient_names,...,folder_path,has_attachments,attachment_count,attachment_names,headers,message_id,in_reply_to,references,source_type,source_file
0,11355263627102,FW: FW: Promab Hybridoma Sequencing. - Laura P...,Judy Tse,judy.tse@promab.com,Martyn Lewis <martyn.lewis@promab.com>,,2015-04-13 09:55:31,"Hi.\n\nWhen you have time, could you please ch...",martyn.lewis@promab.com,Martyn Lewis,...,/Users/promab/anaconda_projects/email_agent/da...,True,1,[91247-BH6 variable region sequence result.docx],X-Antivirus: Avast (VPS 15042801)\nX-Antivirus...,<019801d0760a$a8af0fa0$fa0d2ee0$@promab.com>,,,eml,FW- FW- Promab Hybridoma Sequencing. - Laura P...
1,76313927896914,Re: Custom Rabbit Monoclonal Antibody Service ...,"Gangwani, Laxman",laxman.gangwani@ttuhsc.edu,Martyn Lewis <martyn.lewis@promab.com>,,2015-02-27 14:58:44,"Do you need anything else?\n\nThanks,\n\n--\nL...",martyn.lewis@promab.com,Martyn Lewis,...,/Users/promab/anaconda_projects/email_agent/da...,False,0,[],X-Antivirus: Avast (VPS 15022601)\nX-Antivirus...,<D11644B3.1555A%laxman.gangwani@ttuhsc.edu>,<54EF1E1E.8040806@promab.com>,,eml,Re- Custom Rabbit Monoclonal Antibody Service ...
2,69581235304569,RE: Biocare/Weimin New Quotes,Judy Tse,judy.tse@promab.com,Martyn Lewis <martyn.lewis@promab.com>,,2015-04-23 09:15:09,"Woops, I should mention the target is CD274. T...",martyn.lewis@promab.com,Martyn Lewis,...,/Users/promab/anaconda_projects/email_agent/da...,False,0,[],X-Antivirus: Avast (VPS 15042801)\nX-Antivirus...,<01f601d07de0$aceb4330$06c1c990$@promab.com>,<01ec01d07de0$8e148480$aa3d8d80$@promab.com>,<035301d07d3a$7bf91140$73eb33c0$@promab.com> <...,eml,RE- Biocare-Weimin New Quotes[4].eml
3,132063061765019,Wave Goodbye to Phone-Bills - VoIP Home & Busi...,VoIP Phone Setup,VoIPPhoneSetup@unequalcrbier.xyz,martyn.lewis@promab.com,,2016-01-07 12:51:35,"<!DOCTYPE html PUBLIC ""-//W3C//DTD XHTML 1.0 T...",martyn.lewis@promab.com,,...,/Users/promab/anaconda_projects/email_agent/da...,False,0,[],X-Antivirus: Avast (VPS 16010600)\nX-Antivirus...,<u22444786o22444786.22444786_martyn.lewis@prom...,,,eml,Wave Goodbye to Phone-Bills - VoIP Home & Busi...
4,201597746111193,Promab customer protein project 2015-04-24,promab,promab@163.com,"""judy.tse@promab.com"" <judy.tse@promab.com>, M...","""promab8807@126.com"" <promab8807@126.com>",2015-04-24 00:27:27,"<div style=""line-height:1.7;color:#000000;font...",judy.tse@promab.com; martyn.lewis@promab.com; ...,judy.tse@promab.com; Martyn Lewis; promab8807@...,...,/Users/promab/anaconda_projects/email_agent/da...,True,1,[Promab customer protein project 2015-04-24.xls],X-Antivirus: Avast (VPS 15042801)\nX-Antivirus...,<136b8bd6.1efd9.14cea52f4f4.Coremail.promab@16...,,,eml,Promab customer protein project 2015-04-24.eml


Saved parsed EML files.


In [65]:
print("最早时间:", df_eml_all["date"].min())
print("最晚时间:", df_eml_all["date"].max())

最早时间: 2015-01-09 10:34:03
最晚时间: 2016-09-26 18:00:31


In [66]:
df_eml_all.info()

<class 'pandas.DataFrame'>
RangeIndex: 13941 entries, 0 to 13940
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   id                13941 non-null  int64         
 1   subject           13941 non-null  str           
 2   sender            13941 non-null  str           
 3   sender_email      13941 non-null  str           
 4   to                13941 non-null  str           
 5   cc                13941 non-null  str           
 6   date              13795 non-null  datetime64[us]
 7   body              13941 non-null  str           
 8   recipient_emails  13941 non-null  str           
 9   recipient_names   13941 non-null  str           
 10  source_pst        13941 non-null  str           
 11  folder_path       13941 non-null  str           
 12  has_attachments   13941 non-null  bool          
 13  attachment_count  13941 non-null  int64         
 14  attachment_names  13941 non-null 